# Global Patent Intelligence Data Pipeline

Handles large `.tsv` files using chunking and efficient loading.

In [ ]:
import pandas as pd
import sqlite3
import json
import os

# File paths
patents_path = 'g_patent\\g_patent.tsv'
abstracts_path = 'g_patent_abstract\\g_patent_abstract.tsv'
inventors_path = 'g_inventor_disambiguated\\g_inventor_disambiguated.tsv'
patent_inventor_path = 'g_persistent_inventor\\g_persistent_inventor.tsv'
companies_path = 'g_assignee_disambiguated\\g_assignee_disambiguated.tsv'
patent_company_path = 'g_persistent_assignee\\g_persistent_assignee.tsv'
locations_path = 'g_location_disambiguated\\g_location_disambiguated.tsv'

os.makedirs('working', exist_ok=True)

print("Setup complete")

In [ ]:
def load_large_csv(path, cols, nrows=50000):
    return pd.concat(
        pd.read_csv(
            path,
            sep='\t',
            usecols=cols,
            chunksize=50000,
            nrows=nrows
        ),
        ignore_index=True
    )

print("Loader ready")

In [ ]:
import pandas as pd

sample1 = pd.read_csv(
    patents_path,
    sep='\t',
    nrows=5
)
sample2 = pd.read_csv(
    abstracts_path,
    sep='\t',
    nrows=5
)
sample3 = pd.read_csv(
    inventors_path,
    sep='\t',
    nrows=5
)
sample4 = pd.read_csv(
    patent_inventor_path,
    sep='\t',
    nrows=5
)
sample5 = pd.read_csv(
    companies_path,
    sep='\t',
    nrows=5
)
sample6 = pd.read_csv(
    patent_company_path,
    sep='\t',
    nrows=5
)
sample7 = pd.read_csv(
    locations_path,
    sep='\t',
    nrows=5
)

print(sample1.columns.tolist())
print(sample2.columns.tolist())
print(sample3.columns.tolist())
print(sample4.columns.tolist())
print(sample5.columns.tolist())
print(sample6.columns.tolist())
print(sample7.columns.tolist())


In [ ]:
patents = load_large_csv(
    patents_path,
    ['patent_id','patent_title','patent_date']
)

abstracts = load_large_csv(
    abstracts_path,
    ['patent_id','patent_abstract']
)

inventors = load_large_csv(
    inventors_path,
    [
        'patent_id',
        'inventor_id',
        'disambig_inventor_name_first',
        'disambig_inventor_name_last',
        'location_id'
    ]
)

companies = load_large_csv(
    companies_path,
    [
        'patent_id',
        'assignee_id',
        'disambig_assignee_organization'
    ]
)

locations = load_large_csv(
    locations_path,
    ['location_id','disambig_country']
)

print("All data loaded correctly")

In [ ]:
# Rename
locations.rename(columns={'disambig_country':'country'}, inplace=True)

companies.rename(columns={
    'assignee_id':'company_id',
    'disambig_assignee_organization':'name'
}, inplace=True)

# -------------------------
# PATENTS
patents_clean = patents.merge(abstracts, on='patent_id', how='left')

patents_clean['year'] = pd.to_datetime(
    patents_clean['patent_date'],
    errors='coerce'
).dt.year

patents_clean['year'] = patents_clean['year'].fillna(0).astype(int)

# -------------------------
# INVENTORS
inventors['name'] = (
    inventors['disambig_inventor_name_first'].fillna('') + ' ' +
    inventors['disambig_inventor_name_last'].fillna('')
)

inventors_clean = inventors.merge(
    locations,
    on='location_id',
    how='left'
)[['inventor_id','name','country']]

inventors_clean = inventors_clean.drop_duplicates(subset=['inventor_id'])

# -------------------------
# COMPANIES
companies_clean = companies[['company_id','name']].dropna()
companies_clean = companies_clean.drop_duplicates(subset=['company_id'])

print("Cleaning complete")

In [ ]:
# patent → inventor
rel_inventor = inventors[['patent_id','inventor_id']]

# patent → company
rel_company = companies[['patent_id','company_id']]

# combine
relationships = rel_inventor.merge(
    rel_company,
    on='patent_id',
    how='inner'
)

relationships = relationships.dropna()

print("Relationships created")

In [ ]:
patents_clean.to_csv('working\\clean_patents.csv', index=False)
inventors_clean.to_csv('working\\clean_inventors.csv', index=False)
companies_clean.to_csv('working\\clean_companies.csv', index=False)
relationships.to_csv('working\\relationships.csv', index=False)

print('Saved cleaned files')

In [ ]:
conn = sqlite3.connect('working/patents.db')

patents_clean.to_sql('patents', conn, if_exists='replace', index=False)
inventors_clean.to_sql('inventors', conn, if_exists='replace', index=False)
companies_clean.to_sql('companies', conn, if_exists='replace', index=False)
relationships.to_sql('relationships', conn, if_exists='replace', index=False)

print("Database ready")

In [ ]:
top_inventors = pd.read_sql_query("""
SELECT i.name, COUNT(*) AS total_patents
FROM relationships r
JOIN inventors i ON r.inventor_id = i.inventor_id
GROUP BY i.name
ORDER BY total_patents DESC
LIMIT 10
""", conn)

top_inventors

In [ ]:
top_companies = pd.read_sql_query("""
SELECT c.name, COUNT(*) AS total_patents
FROM relationships r
JOIN companies c ON r.company_id = c.company_id
GROUP BY c.name
ORDER BY total_patents DESC
LIMIT 10
""", conn)

top_companies

In [ ]:
top_countries = pd.read_sql_query("""
SELECT country, COUNT(*) AS total
FROM inventors
GROUP BY country
ORDER BY total DESC
LIMIT 10
""", conn)

top_countries

In [ ]:
trends = pd.read_sql_query("""
SELECT year, COUNT(*) AS total_patents
FROM patents
GROUP BY year
ORDER BY year
""", conn)

trends

In [ ]:
joined_data = pd.read_sql_query("""
SELECT p.patent_title, i.name, c.name
FROM relationships r
JOIN patents p ON r.patent_id = p.patent_id
JOIN inventors i ON r.inventor_id = i.inventor_id
JOIN companies c ON r.company_id = c.company_id
LIMIT 20
""", conn)

joined_data

In [ ]:
cte_query = pd.read_sql_query("""
WITH inventor_counts AS (
    SELECT inventor_id, COUNT(*) AS total
    FROM relationships
    GROUP BY inventor_id
)
SELECT *
FROM inventor_counts
WHERE total > 5
ORDER BY total DESC
""", conn)

cte_query

In [ ]:
ranking = pd.read_sql_query("""
SELECT i.name,
       COUNT(*) AS total,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS rank
FROM relationships r
JOIN inventors i ON r.inventor_id = i.inventor_id
GROUP BY i.name
LIMIT 10
""", conn)

ranking

In [ ]:
# Save reports
top_inventors.to_csv('working\\top_inventors.csv', index=False)
top_companies.to_csv('working\\top_companies.csv', index=False)
top_countries.to_csv('working\\country_trends.csv', index=False)


In [ ]:
report = {
    "total_patents": int(len(patents_clean)),
    "top_inventors": top_inventors.to_dict(orient='records'),
    "top_companies": top_companies.to_dict(orient='records'),
    "top_countries": top_countries.to_dict(orient='records')
}

with open('working/report.json', 'w') as f:
    json.dump(report, f, indent=4)